## Data preprocessing


In [1]:
import json
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from utils import load_data, create_embedding_matrix, preprocess_text, to_padding, pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support


In [2]:
class MyTokenizer:
    def __init__(self):
        self.word_index = {"<PAD>": 0, "<UNK>": 1}
        self.idx_to_token = {0: "<PAD>", 1: "<UNK>"}
        
    def fit_on_texts(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word_index:
                    self.word_index[word] = len(self.word_index)
                    self.idx_to_token[self.word_index[word]] = word

    def text_to_sequences(self, text):
        return [self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()]
    
    def texts_to_sequences(self, texts):
        return [[self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()] for text in texts]


    def encode(self, text, max_length):
        tokens = self.text_to_sequences(text)
        if len(tokens) < max_length:
            tokens += [self.word_index["<PAD>"]] * (max_length - len(tokens))
        else:
            tokens = tokens[:max_length]
        return tokens
    
    def __call__(self, claims, evidences, max_length=512, return_tensors="pt"):
        self.fit_on_texts([claims, evidences])
        encoded_claims = self.encode(claims, max_length)
        encoded_evidences = self.encode(evidences, max_length)
        if return_tensors == "pt":
            return {
                "input_ids": torch.tensor([encoded_claims, encoded_evidences], dtype=torch.long)
            }
        return {"input_ids": [encoded_claims, encoded_evidences]}


In [3]:
# nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')  


In [48]:
# Load filtered evidences
filtered_evidences = load_data("data/curated/climate_dic.json")
filtered_evidence_map = {eid: preprocess_text(text, stemmer, stop_words) for eid, text in filtered_evidences.items()}
with open("filtered_evidence_map.json", "w") as f:
    json.dump(filtered_evidence_map, f)

In [4]:
from sklearn.model_selection import train_test_split

claim_ids = []
for claim_id, claim_details in train_claims_data.items():
	claim_ids.append(claim_id)

# split the claims_df into training and test sets
train, test = train_test_split(claim_ids, test_size=0.2, random_state=42)
len(train)

982

In [5]:
# train_data_for_dataframe = []
# test_data_for_dataframe = []
# evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

# for claim_id, claim_details in train_claims_data.items():
# 	claim_text = preprocess_text(claim_details['claim_text'], stemmer, stop_words)
# 	claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

# 	# Add positive examples
# 	for eid in claim_evidences:
# 		evidence_text = evidence_map.get(eid, "NULL")  
# 		if evidence_text != "NULL":
# 			data = {
# 				'claim': claim_text,
# 				'evidence': evidence_text,
# 				'label': 1  # Label as relevant
# 			}
# 			if claim_id in train:
# 				train_data_for_dataframe.append(data)
# 			else:
# 				test_data_for_dataframe.append(data)

# 	# Add negative examples
# 	num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
# 	negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
# 	for eid in negative_samples:
# 		evidence_text = evidence_map[eid]
# 		data = {
# 			'claim': claim_text,
# 			'evidence': evidence_text,
# 			'label': 0  # Label as not relevant
# 		}
# 		if claim_id in train:
# 				train_data_for_dataframe.append(data)
# 		else:
# 			test_data_for_dataframe.append(data)

# train_df = pd.DataFrame(train_data_for_dataframe)
# test_df = pd.DataFrame(test_data_for_dataframe)

# train_df.to_csv('train_data.csv', index=False)
# test_df.to_csv('test_data.csv', index=False)
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

train_df = train_df.dropna()
test_df = test_df.dropna()

print(train_df.head(10))


                                               claim  \
0  not onli is there no scientif evid that co2 is...   
1  not onli is there no scientif evid that co2 is...   
2  not onli is there no scientif evid that co2 is...   
3  not onli is there no scientif evid that co2 is...   
4  not onli is there no scientif evid that co2 is...   
5  not onli is there no scientif evid that co2 is...   
6  el niño drove record high in global temperatur...   
7  el niño drove record high in global temperatur...   
8  el niño drove record high in global temperatur...   
9  el niño drove record high in global temperatur...   

                                            evidence  label  
0  plant grow much percent faster concentr ppm co...      1  
1  high concentr time atmospher concentr greater ...      1  
2  higher carbon dioxid concentr favour affect pl...      1  
3  closest tube station sloan squar north along l...      0  
4       air also run luxuri flow hair club scientist      0  
5   road st

In [6]:
tokenizer = MyTokenizer()
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index, y_claims_data, y_sents_data, y_labels = to_padding(train_df, test_df, tokenizer)

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

vocab_size_claims = len(x_claims_word_index) + 2  # +1 for padding, +1 for <UNK>
vocab_size_evidences = len(x_sents_word_index) + 2

Max length: 66
Max length: 177
x claim word index  14768
x sent word index  14768


In [7]:
class EvidenceDataset(Dataset):
    def __init__(self, claims, evidences, labels):
        self.claims = claims
        self.evidences = evidences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        claim = torch.tensor(self.claims[idx], dtype=torch.long)
        evidence = torch.tensor(self.evidences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return {
            'claims': claim,
            'evidences': evidence,
            'labels': label
        }

# Assuming x_claim, y_claims_data, etc. are numpy arrays or lists of integers
train_dataset = EvidenceDataset(x_claim, x_sents, x_labels)
test_dataset = EvidenceDataset(y_claims_data, y_sents_data, y_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


## Creating the Embedding Matrix

In [8]:
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

In [9]:
embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

embed_matrix_claim shape  (14770, 300)
embed_matrix_evidence shape  (14770, 300)


## Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [10]:
class EvidenceModel(nn.Module):
    def __init__(self, vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence):
        super(EvidenceModel, self).__init__()
        self.embedding_claims = nn.Embedding(vocab_size_claims, embed_dim_claim)
        self.lstm_claims_1 = nn.LSTM(embed_dim_claim, 256, batch_first=True)
        self.lstm_claims_2 = nn.LSTM(256, 16, batch_first=True)
        self.batch_norm_claims = nn.BatchNorm1d(16)
        
        self.embedding_evidences = nn.Embedding(vocab_size_evidences, embed_dim_evidence)
        self.lstm_evidences_1 = nn.LSTM(embed_dim_evidence, 256, batch_first=True)
        self.lstm_evidences_2 = nn.LSTM(256, 64, batch_first=True)
        self.batch_norm_evidences = nn.BatchNorm1d(64)
        
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(16 + 64, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, claims_input, evidences_input):
        embedded_claims = self.embedding_claims(claims_input)
        lstm_out_claims, _ = self.lstm_claims_1(embedded_claims)
        lstm_out_claims, _ = self.lstm_claims_2(lstm_out_claims)
        lstm_out_claims = lstm_out_claims[:, -1, :]  # Get the last output for each sequence
        norm_claims = self.batch_norm_claims(lstm_out_claims)
        
        embedded_evidences = self.embedding_evidences(evidences_input)
        lstm_out_evidences, _ = self.lstm_evidences_1(embedded_evidences)
        lstm_out_evidences, _ = self.lstm_evidences_2(lstm_out_evidences)
        lstm_out_evidences = lstm_out_evidences[:, -1, :]  # Get the last output for each sequence
        norm_evidences = self.batch_norm_evidences(lstm_out_evidences)
        
        concatenated = torch.cat((norm_claims, norm_evidences), dim=1)
        concatenated = self.dropout(concatenated)
        concatenated = self.fc1(concatenated)
        concatenated = self.relu(concatenated)
        output = self.fc2(concatenated)
        output = self.sigmoid(output)
        return output

def train_model(model, train_loader, val_loader, epochs, device):
    model_path = 'lstm_evidence_retrieval.pth'
    best_val_loss = float('inf')
    patience = 2
    trigger_times = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)
        print(f'Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

def predict(model, data_loader, device):
    model.eval() 
    predictions = []
    with torch.no_grad():  
        for batch in data_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            outputs = model(claims, evidences)
            predictions.append(outputs.cpu())  # Move predictions to CPU
            
    # Concatenate the list of tensors into a single tensor
    predictions = torch.cat(predictions, dim=0)
    return predictions


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EvidenceModel(vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [12]:

# # Train the model
# epochs = 60
# train_model(model, train_loader, test_loader, epochs, device)

## Test model

In [13]:
# Load the best model
model.load_state_dict(torch.load('lstm_evidence_retrieval.pth'))

# Test loader setup like train_loader and val_loader
test_loss = evaluate(model, test_loader, device)
print("Test loss", test_loss)

# Prediction and performance metrics
y_pred = []
y_true = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        claims = batch['claims'].to(device)
        evidences = batch['evidences'].to(device)
        labels = batch['labels'].cpu().numpy()
        outputs = model(claims, evidences).cpu().numpy().round()
        y_pred.extend(outputs)
        y_true.extend(labels)

# Calculate precision, recall, and F-score
precision, recall, fscore, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
print("Score of LSTM", {'precision': precision, 'recall': recall, 'fscore': fscore})


Test loss 0.3441122936851838
Score of LSTM {'precision': 0.7832661290322581, 'recall': 0.9640198511166254, 'fscore': 0.864293659621802}


## Predict dev-claims

In [14]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'], stemmer, stop_words)
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'original_claim': claim_details['claim_text'],
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

,claim_id,claim,original_claim,evidence
0,claim-752,south australia ha the most expens electr in t...,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]"
1,claim-375,when 3 per cent of total annual global emiss o...,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,thi mean that the world is now 1c warmer than ...,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]"
3,claim-871,as it happen zika may also be a good model of ...,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,greenland ha onli lost a tini fraction of it i...,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...,...
149,claim-2400,suddenli label co2 as a pollut is a disservic ...,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,after a natur orbit driven warm atmospher carb...,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,mani of the world s coral reef are alreadi bar...,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,a recent studi led by lawrenc livermor nation ...,A recent study led by Lawrence Livermore Natio...,[evidence-660755]


In [15]:
with open('tokenizer_claims.pickle', 'rb') as handle:
	claims_tokenizer = pickle.load(handle)

with open('tokenizer_evidence.pickle', 'rb') as handle:
	sents_tokenizer = pickle.load(handle)

max_claims_length = 70
max_sents_length = 180

In [51]:
dev_claims = claims_tokenizer.texts_to_sequences(dev_claims_df["claim"].tolist())
dev_sents = sents_tokenizer.texts_to_sequences(filtered_evidence_map.values())

dev_claims = pad_sequences(dev_claims, maxlen=max_claims_length)
dev_sents = pad_sequences(dev_sents, maxlen=max_sents_length)
print ("dev claims ", dev_claims.shape)
print ("dev sents ", dev_sents.shape)

dev claims  (154, 70)
dev sents  (298688, 180)


In [55]:

model = model.to(device)  
model.eval()  

dev_claims = torch.tensor(dev_claims).to(device)  
dev_sents = torch.tensor(dev_sents).to(device)
claim_ids = dev_claims_df["claim_id"].tolist()
evidence_ids = list(evidence_map.keys())

batch_size = 1024  
threshold = 0.98
top_k = 5
all_predictions = []

for i in range(dev_claims.shape[0]):
    claim_row = dev_claims[i].unsqueeze(0)
    claim_id = claim_ids[i]
    # qualified_evidence_ids = []
    top_evidence_ids = []

    # Aggregate scores and corresponding evidence IDs across all batches
    all_scores = []
    all_batch_evidence_ids = []

    # Process in batches
    for j in range(0, dev_sents.shape[0], batch_size):
        end = min(j + batch_size, dev_sents.shape[0])
        batch_sents = dev_sents[j:end].to(device)
        # replicated_claims_batch = replicated_claims[:end-j].to(device)  
        replicated_claims_batch = claim_row.repeat(batch_sents.shape[0], 1).to(device)

        with torch.no_grad():
            outputs = model(replicated_claims_batch, batch_sents).squeeze(1)

            all_scores.extend(outputs.cpu().numpy())  # Collect scores
            all_batch_evidence_ids.extend(evidence_ids[j:end])

            # predictions = outputs > threshold
            # print("Y_PREDICT: ", outputs.cpu().numpy())

            # Collect evidence IDs for this batch where predictions are true
            # true_indices = predictions.cpu().nonzero(as_tuple=False).squeeze(1)
            # batch_evidence_ids = evidence_ids[j:end]  
            # qualified_evidence_ids.extend([batch_evidence_ids[idx] for idx in true_indices])

    # Store results for this claim
    if all_scores:
        # Get indices of the top k scores
        top_indices = sorted(range(len(all_scores)), key=lambda x: all_scores[x], reverse=True)[:top_k]
        top_evidence_ids = [all_batch_evidence_ids[index] for index in top_indices]

    # Store results for this claim
    if top_evidence_ids:
        all_predictions.append({
            "claim_id": claim_id,
            "evidences_id": top_evidence_ids
        })
    # if qualified_evidence_ids:
    #     all_predictions.append({
    #         "claim_id": claim_id,
    #         "evidences_id": qualified_evidence_ids
    #     })
    break

results_df = pd.DataFrame(all_predictions)


C:\Users\Clare\AppData\Local\Temp\ipykernel_37312\1754254489.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dev_claims = torch.tensor(dev_claims).to(device)
C:\Users\Clare\AppData\Local\Temp\ipykernel_37312\1754254489.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dev_sents = torch.tensor(dev_sents).to(device)


In [56]:
len(results_df['evidences_id'][0])

5

In [58]:
results_df['evidences_id'][0]

['evidence-38762',
 'evidence-65298',
 'evidence-82790',
 'evidence-88584',
 'evidence-145404']

In [37]:
from itertools import product

claims = dev_claims_df['original_claim'].tolist()  
claim_ids = dev_claims_df['claim_id'].tolist()  
evidence_texts = list(evidence_map.values())  
evidence_ids = list(evidence_map.keys())  

# Generate the Cartesian product of claims and evidences
crossed_claims, crossed_evidences, crossed_claim_ids, crossed_evidence_ids, crossed_claims_text = zip(*[
    (claim, evidence, claim_id, evidence_id, claim_text)
    for (claim_text, claim_id, claim), (evidence, evidence_id) in product(zip(claims, claim_ids, dev_claims), zip(evidence_texts, evidence_ids))
])

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000148E1DB8940>>
Traceback (most recent call last):
  File "c:\Users\Clare\miniconda3\envs\nlp\lib\site-packages\ipykernel\ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [21]:
class TestData(Dataset):
    def __init__(self, claims, evidences, claim_ids, eids):
        self.claims = claims
        self.evidences = evidences
        self.claim_ids = claim_ids  
        self.eids = eids

    def __len__(self):
        return len(self.claims)

    def __getitem__(self, idx):
        claim = torch.tensor(self.claims[idx], dtype=torch.long)
        evidence = torch.tensor(self.evidences[idx], dtype=torch.long)
        claim_id = self.claim_ids[idx]
        evidence_id = self.eids[idx]
        return {
            'claim': claim,
            'evidence': evidence,
            'claim_id': claim_id,
            'evidence_id': evidence_id
        }


# dev_dataset = TestData(dev_claims, dev_sents, dev_claims_df["original_claim"].tolist(), dev_claims_df["claim_id"].tolist(), list(evidence_map.keys()),  list(evidence_map.values()))

dev_dataset = TestData(crossed_claims, crossed_evidences, crossed_claim_ids, crossed_evidence_ids)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)


In [16]:
model.load_state_dict(torch.load('lstm_evidence_retrieval.pth'))

<All keys matched successfully>

In [ ]:
# # Make predictions
# model.load_state_dict(torch.load('lstm_evidence_retrieval.pth'))
# model_predictions = predict(model, test_loader, device)

# print("Predictions:", model_predictions)

In [ ]:
# def predict_and_filter(model, data_loader, device, threshold=0.5):
#     model.eval()
#     filtered_claims = []
#     filtered_evidences = []
#     with torch.no_grad():
#         for batch in data_loader:
#             claims = batch['claim_id'].to(device)
#             evidences = batch['evidence_id'].to(device)
#             outputs = model(claims, evidences).squeeze(1)
#             mask = outputs > threshold  # Get the mask where predictions exceed the threshold

#             # Append texts that meet the condition
#             filtered_claims.extend([text for text, m in zip(batch['claims_text'], mask) if m])
#             filtered_evidences.extend([text for text, m in zip(batch['evidences_text'], mask) if m])

#     return filtered_claims, filtered_evidences

# filtered_claims, filtered_evidences = predict_and_filter(model, test_loader, device, threshold=0.7)
# for claim, evidence in zip(filtered_claims, filtered_evidences):
#     print("Claim:", claim)
#     print("Evidence:", evidence)
#     print("----------")

In [30]:
from collections import defaultdict
def predict_and_collect(model, data_loader, device, threshold=0.5):

    model.eval()
    results = defaultdict(lambda: {
        'claim_id': None, 'claim_text_raw': None, 'evidence_id': [], 'evidence_texts': []
    })

    with torch.no_grad():
        for batch in data_loader:
            claims = batch['claim'].to(device)
            evidences = batch['evidence'].to(device)
            outputs = model(claims, evidences).squeeze(1)
            predictions = outputs > threshold  # Get the mask where predictions exceed the threshold

            for i, pred in enumerate(predictions):
                if pred:  # If the prediction is above the threshold
                    claim_id = batch['claim_id'][i]
                    evidence_id = batch['evidence_id'][i]
                    claim_text = batch['claim_text'][i]
                    evidence_text = batch['evidence_text'][i]

                    if results[claim_id]['claim_id'] is None:
                        results[claim_id]['claim_id'] = claim_id
                        results[claim_id]['claim_text_raw'] = claim_text

                    results[claim_id]['evidence_id'].append(evidence_id)
                    results[claim_id]['evidence_texts'].append(evidence_text)

    # Convert the results dictionary to a list for easier processing later
    results_list = [value for key, value in results.items()]
    return results_list


# Example of using the function
output = predict_and_collect(model, dev_loader, device, threshold=0.5)
print(output)


[{'claim_id': 'claim-752', 'claim_text_raw': '[South Australia] has the most expensive electricity in the world.', 'evidence_id': ['evidence-0'], 'evidence_texts': ['john bennet law english entrepreneur agricultur scientist']}, {'claim_id': 'claim-1607', 'claim_text_raw': "CO2 limits won't cool the planet.", 'evidence_id': ['evidence-5'], 'evidence_texts': ['peak wind mph minimum pressur mbar hpa inhg florenc strongest storm atlant hurrican season']}, {'claim_id': 'claim-761', 'claim_text_raw': '[Riebesell] is a world authority on the topic and has typically communicated cautiously about the effects of acidification.', 'evidence_id': ['evidence-6'], 'evidence_texts': ['current professor piano univers wisconsin madison sinc august']}, {'claim_id': 'claim-1718', 'claim_text_raw': 'The actual data show high northern latitudes are warmer today than in 1940.', 'evidence_id': ['evidence-7'], 'evidence_texts': ['addit known tangibl risk unforese black swan extinct event may occur present addi

In [31]:

results_df = pd.DataFrame(output)
results_df.rename(columns={'claim_id': 'claim_id', 'claim_text_raw': 'claim_text_raw',
                           'evidence_id': 'evidence_id', 'evidence_texts': 'evidence_texts'}, inplace=True)

# Display the DataFrame
print(results_df.head())


     claim_id                                     claim_text_raw  \
0   claim-752  [South Australia] has the most expensive elect...   
1  claim-1607                  CO2 limits won't cool the planet.   
2   claim-761  [Riebesell] is a world authority on the topic ...   
3  claim-1718  The actual data show high northern latitudes a...   
4  claim-2796  Had he used the currently accepted value of ap...   

     evidence_id                                     evidence_texts  
0   [evidence-0]  [john bennet law english entrepreneur agricult...  
1   [evidence-5]  [peak wind mph minimum pressur mbar hpa inhg f...  
2   [evidence-6]  [current professor piano univers wisconsin mad...  
3   [evidence-7]  [addit known tangibl risk unforese black swan ...  
4  [evidence-10]  [matroid dual go back origin paper hassler whi...  


In [32]:
aggregated_df = results_df.groupby('claim_id').agg({
    'claim_text_raw': 'first',  # Assuming all entries for the same claim_id have the same claim_text_raw
    'evidence_id': lambda x: [item for sublist in x for item in sublist],  
    'evidence_texts': lambda x: [item for sublist in x for item in sublist]  
}).reset_index()

print(aggregated_df.head())

     claim_id                                     claim_text_raw  \
0  claim-1021  The corals may save themselves, as many other ...   
1  claim-1040  While evidence that the earth’s orbital variat...   
2  claim-1167  Lake-bottom sediments in Florida tell us that ...   
3  claim-1219  Global warming is driving major melting on the...   
4  claim-1292  Any reasonable person can recognize both posit...   

      evidence_id                                     evidence_texts  
0  [evidence-153]                 [club degre branch degre ministri]  
1  [evidence-108]  [time known year consulship lasciviu atiliu le...  
2  [evidence-124]  [station own combin commun sign air rhythmic c...  
3   [evidence-12]  [current aida individu world champion constant...  
4   [evidence-35]  [jack jump also known skibock europ action spo...  


In [33]:
aggregated_df['evidence_id_length'] = aggregated_df['evidence_id'].apply(len)
average_length = aggregated_df['evidence_id_length'].mean()
print(f"The average length of the evidence_id list is: {average_length}")


The average length of the evidence_id list is: 1.0


In [35]:
len(aggregated_df)

44